# GDELT/EDGAR risk labeling - REAL run (8,360 rows, prompt v2)

**Before running:** GPU T4 x2 + Internet On, HF_TOKEN secret (accept Llama 3.1 license on HF once), upload **labeling_batch_2026-07-22.csv**. Run top to bottom. Output: labels_2026-07-22.csv.

In [ ]:
!pip -q install -U vllm openai
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret - add it in Add-ons > Secrets. Detail:", e)

In [ ]:
import csv
import glob

LABELS = ("positive", "negative", "neutral")
ADAPTERS = {"gdelt": ("title",), "edgar": ("title", "text_excerpt")}


def text_for(r):
    return "\n".join(str(r[f]) for f in ADAPTERS[r["source"]] if r.get(f)).strip()


def parse_label(raw):
    t = raw.strip().lower()
    return next((l for l in LABELS if l in t), None)


PROMPT = r"""<!-- prompt_version: v2 -->
You label news articles about US banks by the RISK DIRECTION they imply for
the bank — not by the article's emotional tone.

- negative: implies the bank's risk is RISING / health worsening (losses,
  deposit outflows, enforcement actions — including entering a formal or
  written agreement, consent order, or cease-and-desist with a regulator
  (OCC, Fed, FDIC) — lawsuits, risk/finance executive exits, "exploring
  strategic alternatives", and other euphemistic distress signals).
- positive: implies risk FALLING / health improving (capital raises, consent
  orders lifted, earnings improvement, rating upgrades).
- neutral: no clear risk direction (routine announcements, product launches,
  branch openings, sponsorships, incidental mentions).

Examples:
Article: "Regional Bank reports third straight quarter of deposit outflows"
Label: negative
Article: "Community Bancorp says it is exploring strategic alternatives"
Label: negative
Article: "Pinnacle Bank's chief risk officer resigns after two years"
Label: negative
Article: "Coastal Trust enters a formal written agreement with the OCC over its BSA/AML program"
Label: negative
Article: "Federal Reserve lifts consent order against Midwest Bank"
Label: positive
Article: "Summit Bank raises $500M in capital, lifting its Tier 1 ratio"
Label: positive
Article: "Coastal Bank opens three new branches in the metro area"
Label: neutral

Now label this article. Answer with exactly one word: positive, negative,
or neutral.

Article: "{{ARTICLE}}"
Label:
"""

INPUT_CSV = glob.glob("/kaggle/input/**/labeling_batch_2026-07-22.csv", recursive=True)[
    0
]
rows = list(csv.DictReader(open(INPUT_CSV, encoding="utf-8")))
prompts = [PROMPT.replace("{{ARTICLE}}", text_for(r)) for r in rows]
print(len(rows), "rows from", INPUT_CSV)

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    quantization="awq",
    max_model_len=4096,
)
tok = llm.get_tokenizer()
texts = [
    tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True
    )
    for p in prompts
]
params = SamplingParams(temperature=0, max_tokens=4)
outs = llm.generate(texts, params)
labels = [parse_label(o.outputs[0].text) for o in outs]

In [ ]:
import json
from collections import Counter

meta = json.dumps(
    {
        "model": "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
        "quantization": "awq",
        "prompt_version": "v2",
        "run_date": "2026-07-22",
    }
)
valid = [(r, l) for r, l in zip(rows, labels) if l in LABELS]
skipped = len(rows) - len(valid)
with open("/kaggle/working/labels_2026-07-22.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["raw_item_id", "label", "model_meta"])
    w.writeheader()
    for r, l in valid:
        w.writerow({"raw_item_id": r["raw_item_id"], "label": l, "model_meta": meta})
print("written:", len(valid), "| skipped (unparseable):", skipped)
print(Counter(l for _, l in valid))